# Replication of
## *Smart Green Nudging: Reducing Product Returns Through Digital Footprints and Causal Machine Learning*
### Moritz von Zahn et al. (2025), *Marketing Science*

---

## Introduction


This replication focuses on the **preliminary estimation stage** of the paper, covering the Average Treatment Effect (ATE) and policy evaluation via the IPS estimator. It aims to remain as faithful as possible to the original empirical structure, allowing for direct comparison of results.

The primary deviations from the original implementation involve updating package dependencies for compatibility with current software environments and one minor enhancement to the GATE plot (sample counts added for clarity). All modifications are documented in the Implementation Adjustments section.

This replication uses the dataset provided in the supplemental materials, rather than the original proprietary experimental dataset used in the paper. As discussed in the Results section, this distinction has implications for direct numerical comparability. No preprocessing beyond reading the dataset into Python was applied. All variables were used as provided in the supplemental data files.

---

## Motivation

Product returns have increased substantially with the rise of e-commerce. Unlike in-person shopping, online purchasing removes the ability to physically inspect products prior to purchase, leading consumers to rely more heavily on trial-and-error behavior.

While this increases convenience, it generates significant costs for firms and contributes to environmental externalities. As noted in the paper, product returns:

“lead to increased greenhouse gas emissions and the squandering of natural resources.”

To address this, the authors propose a behavioral intervention based on green nudging, where consumers are informed about the environmental consequences of returns. This builds on prior evidence that non-monetary incentives—such as environmental salience—can influence decision-making.

---

## Experimental Design

The authors implement a randomized field experiment with
$n = 117,304$
customers on a European online fashion retailer's platform.

Customers are randomly assigned to:
- **Control group:** no intervention
- **Single nudge:** one environmental message
- **Dual (smart) nudge:** multiple prompts with targeting

Randomization ensures 
$T \perp (Y(0), Y(1))$ 
allowing unbiased estimation of causal effects.

### Average Treatment Effect (ATE)

$$\hat{\tau}_{ATE} = E[Y \mid T=1] - E[Y \mid T=0]$$

The paper finds that green nudging reduces return rates by approximately **2.6 percentage points**, with no negative effect on sales.

### Heterogeneous Treatment Effects (CATE)

$$\tau(x) = E[Y(1) - Y(0) \mid X = x]$$

where $X$ represents a customer's digital footprint. This allows the firm to identifyt which customers should recieve the targeted nudge

---

## Digital Footprint Variables

**Initial Cart:** 
- Cart value (numerical)

- Number of products (numerical)

- Number of eco-friendly products (numerical)

- Contains duplicate products (binary)

**Browsing Behavior:** 

- Women category (binary)

- Kids category (binary)

- Sale category (binary)

**Contextual Features** *(not fully present in provided dataset):* 
- Weekday (categorical)

- Internet provider (categorical)

- Browser (categorical)

- Geolocation (categorical)

---

## Causal Machine Learning Approach

The authors implement a **causal forest** in a Double Machine Learning (DML) framework. Unlike a standard prediction forest which minimizes $\min E[(Y - \hat{Y})^2]$, a causal forest partitions the data to maximize treatment effect heterogeneity and estimates treatment effects within leaves:

$$\hat{\tau}(x) = \frac{1}{B} \sum_{b=1}^{B} \hat{\tau}_b(x)$$

This method is used because it identifies which customers are most responsive to intervention, allows for non-linear interactions among a large number of covariates (digital footprint and cart data), and uses a double machine learning framework to reduce bias in treatment effect estimation.


### Policy Targeting (Smart Nudging)

$$\pi(x_i) = \begin{cases} 1 & \text{if } \hat{\tau}(x_i) < 0 \\ 0 & \text{otherwise} \end{cases}$$



Since the outcome is returns, a negative $\hat{\tau}(x)$ means treatment reduces returns — these individuals are targeted. In code, this is implemented as:

```python
testing_frame['treat_policy'] = np.where(testing_frame['effect'] < 0, 1, 0)
```

### Off-Policy Evaluation (IPS Estimator)

$$\hat{R}_{IPS}(\pi) = \frac{1}{N} \sum_{i=1}^{N} \left( \frac{1 - w_i}{1 - \hat{e}(x_i)} (1 - \pi(x_i)) y_{i,0} + \frac{w_i}{\hat{e}(x_i)} \pi(x_i) y_{i,1} \right)$$

This estimator reweights observed outcomes to estimate the expected outcome under a counterfactual policy.

The **random baseline** treats a random subset of customers using no covariate information, matching the treatment share of the learned policy. A policy that consistently outperforms the random baseline demonstrates that the model captures meaningful heterogeneity in treatment effects.

The causal forest is estimated using `CausalForestDML` from the `econml` library. The key modelling choices are:
- `discrete_treatment=True` — treatment is binary
- `discrete_outcome=True` — return outcome is binary
- No `random_state` is set, matching the original implementation 
```python
est = CausalForestDML(discrete_treatment=True, discrete_outcome=True)
est.fit(Y=y_train, T=t_train, X=x_train)
```

Full training code including hyperparameter tuning and loading is in `src/cml_training.py`.

In [ ]:
from src.cml_training import train_model

train_model()

---

## Model Evaluation

### Implementation Adjustments

Several modifications were required to produce the plots in a modern Windows environment. These are documented here for full transparency:

1. **`econml` compatibility patch** — `econml/utilities.py` line 1519: `from pkg_resources import parse_version` replaced with `from packaging.version import Version as parse_version` for Python 3.11+ compatibility.
2. **Matplotlib backend** — `matplotlib.use('Agg')` forces a non-interactive backend, as the default backend fails silently when running from a terminal without a GUI display context on Windows.
3. **LaTeX rendering** — `plt.rcParams['text.usetex'] = False` disables the LaTeX requirement introduced by the `science` style, improving portability across systems without LaTeX installed.
4. **Missing dependency** — `scienceplots==2.2.0` was added to `requirements.txt` as it was not included in the original package.

Full evaluation code is in `src/cml_evaluation.py`.

In [ ]:
from src.cml_evaluation import evaluate_model

evaluate_model()

---

## Results

This replication focuses on the preliminary estimation stage, covering the Average Treatment Effect (ATE) and policy evaluation via the IPS estimator. The evaluation dataset consists of $N = 10{,}000$ observations, of which $N_{treated} = 6{,}398$ received treatment under the targeted policy.

### Group Average Treatment Effects (GATE)

The GATE plot shows the average treatment effect for each policy group:
- $\pi(x_i) = 1$ ($n = 6{,}398$): customers the model recommends **treating** — negative ATE, meaning treatment reduces returns
- $\pi(x_i) = 0$ ($n = 3{,}602$): customers the model recommends **not treating** — ATE near zero or slightly positive

This confirms the model successfully identifies individuals for whom treatment reduces return rates. Sample counts were added to the replicated figure for clarity, as they are not shown in the original.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

img_original = mpimg.imread("plots/gate_plot.pdf")
img_reproduced = mpimg.imread("plots/gate_plot_produced.png")

axes[0].imshow(img_original)
axes[0].axis("off")
axes[0].set_title("Original (installed)", fontsize=14)

axes[1].imshow(img_reproduced)
axes[1].axis("off")
axes[1].set_title("Reproduced (with n counts added)", fontsize=14)

plt.suptitle("Figure 1: GATE Plot — Original vs. Reproduced", fontsize=15, y=1.02)
plt.tight_layout()
plt.show()

### Policy Evaluation (IPS Curves)

The performance of the CML policy is evaluated using the IPS estimator across all treatment shares from 0 to 1.

The replicated IPS curve shows:
- The causal forest policy consistently **outperforms the random baseline**
- Mean return rates **decrease as targeting becomes more selective**
- The overall **shape closely matches** the original

From the reproduced plot, returns range from approximately $0.30 \rightarrow 0.27$ as treatment share increases under the learned policy. Compared to the original, the curves are visually very similar in shape and ordering — small deviations appear in levels but not in qualitative conclusions. This suggests that the publicly available dataset differs from the original experimental data used in the paper possibly due to data anonymization or preprocessing modifications.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

img_original = mpimg.imread("plots/ips_plot.pdf")
img_reproduced = mpimg.imread("plots/ips_plot_produced.png")

axes[0].imshow(img_original)
axes[0].axis("off")
axes[0].set_title("Original (installed)", fontsize=14)

axes[1].imshow(img_reproduced)
axes[1].axis("off")
axes[1].set_title("Reproduced", fontsize=14)

plt.suptitle("Figure 2: IPS Curve — Original vs. Reproduced", fontsize=15, y=1.02)
plt.tight_layout()
plt.show()

### Comparison to Published Results

| | Baseline IPS value |
|:---|:---|
| Published paper (Figure 4) | ≈ 0.265 |
| This replication | ≈ 0.29 |

This shift in outcome levels reflects a difference in the underlying dataset, not a failure of replication. Since $\hat{R}_{IPS}(\pi)$ depends directly on $Y$, differences in the dataset shift the curve vertically. The publicly available dataset likely differs from the original experimental data due to anonymization, subsampling, or simulation. Despite this, the replication matches the curve shape, policy ordering, and relative improvement over the random baseline.

---

## Reproducibility and Randomness

A key observation from the replication is that the causal forest implementation does not fix a random seed — `CausalForestDML` is initialized without a `random_state` argument in the original code. As a result, each training run produces slightly different treatment effect estimates $\hat{\tau}(x)$, policy assignments $\pi(x)$, and IPS curve values.

However, despite this non-deterministic variation, the replicated results consistently match the original findings in terms of qualitative patterns and relative performance. The high degree of visual overlap between the installed and reproduced outputs suggests the model is statistically stable — exact numerical replication is not expected, but distributional replication is achieved.

To make future runs fully deterministic, one could set:

```python
est = CausalForestDML(discrete_treatment=True, discrete_outcome=True, random_state=42)
```

This was intentionally not done here in order to preserve fidelity to the original implementation.

---

## Limitations

1. **Dataset:** This replication uses the supplemental dataset rather than the original experimental data. The difference in baseline return rates (≈0.29 vs. ≈0.265) limits direct comparability of absolute values.
2. **Stochastic training:** The absence of a fixed random seed prevents exact numerical replication, though qualitative conclusions remain stable across runs.
3. **Reduced feature set:** The provided dataset omits several categorical variables described in the original study (e.g., browser, location, weekday), potentially limiting the model's ability to capture richer forms of treatment effect heterogeneity.

---

## Conclusion

This replication successfully reproduces the core findings of the original study:

- Treatment effects are heterogeneous across customers
- Targeted policies outperform random assignment
- Causal machine learning improves intervention efficiency by focusing on individuals most responsive to treatment

Although differences in data and stochastic training prevent exact numerical replication, the results confirm the robustness of the methodology and its underlying economic insights.

---
## Reproducibility

The causal forest implementation does not fix a random seed.

As a result:

- Treatment effect estimates vary slightly across runs

- IPS curves shift marginally

However:

- Results remain qualitatively consistent

- The model demonstrates statistical reproducibility, though not exact numerical replication

---
## Limitations and Conclusion

This replication is subject to several limitations. First, it relies on the supplemental dataset rather than the original experimental data, as reflected in the difference in baseline return rates between the replication and published results, which limits direct comparability of absolute values. Second, the causal forest introduces stochastic variation due to the absence of a fixed random seed; while this does not affect qualitative conclusions, it prevents exact numerical replication. Third, the provided dataset has a reduced feature set, omitting some predictors described in the original study, which may limit the model’s ability to fully capture treatment effect heterogeneity. Additionally, the experimental sample is drawn from specific shopping categories, such as Women, Kids, and Sale, which may influence the generalizability of results; for instance, Women and Kids categories are heavily women-oriented, and Sale shoppers may be more price-sensitive. Despite these limitations, the replication successfully reproduces the core findings of the original study: treatment effects are heterogeneous, targeted policies outperform random assignment, and causal machine learning improves intervention efficiency. Overall, differences in data and stochastic training prevent exact numerical replication, but the results confirm the robustness of the methodology and its underlying economic insights.